In [14]:
import json

with open('../sample_data/sample_playlist_data.json') as file:
    data = json.load(file)


In [15]:
playlist_title = data['playlist']['title']
print(playlist_title)

playlist_videos = data['playlist']['videos']
print(playlist_videos)

Games, Beats, and Python
[{'video_id': 'vid_gaming_001', 'title': 'Minecraft Survival: Your First Night', 'description': 'A beginner guide to gathering resources, crafting tools, and surviving the first night.', 'channel_id': 'channel_blockcraft', 'channel_title': 'BlockCraft Academy', 'tags': ['minecraft', 'survival', 'beginner guide', 'crafting'], 'published_at': '2025-03-15T14:00:00Z', 'duration_seconds': 742, 'view_count': 420000, 'like_count': 18600, 'topic_label': 'gaming', 'format': 'tutorial'}, {'video_id': 'vid_gaming_002', 'title': 'Build an Automatic Wheat Farm in Minecraft', 'description': 'Step-by-step redstone tutorial for a compact automatic wheat farm.', 'channel_id': 'channel_blockcraft', 'channel_title': 'BlockCraft Academy', 'tags': ['minecraft', 'redstone', 'automatic farm', 'tutorial'], 'published_at': '2025-06-04T16:30:00Z', 'duration_seconds': 615, 'view_count': 275000, 'like_count': 14200, 'topic_label': 'gaming', 'format': 'tutorial'}, {'video_id': 'vid_gaming_

In [16]:
# find how many videos are in each topic label
topic_labels = {}
for video in playlist_videos:
    key = video['topic_label']
    topic_labels[key] = topic_labels.get(key, 0) + 1
print(topic_labels)

{'gaming': 7, 'music': 3, 'programming': 2}


In [17]:
# find how many channels there are and which one appears the most often
channels = {}
for video in playlist_videos:
    key = video['channel_title']
    channels[key] = channels.get(key, 0) + 1
print(channels)

{'BlockCraft Academy': 4, 'BuildWise': 2, 'Survival Quest': 1, 'Soft Signal': 2, 'Beat Lab Notes': 1, 'Python Path': 2}


In [18]:
# Which videos have missing tags?
videos_with_missing_tags = []
for video in playlist_videos:
    if video['tags'] == None:
        videos_with_missing_tags.append((video['video_id'], video['title']))
        
print(videos_with_missing_tags)

[('vid_gaming_007', 'Understanding Minecraft Enchantments')]


In [19]:
candidate_videos = data['candidate_videos']
playlist_video_ids = {video['video_id'] for video in playlist_videos}
valid_candidate_list = [(video['video_id'], video['title']) 
                        for video in candidate_videos if video['video_id'] not in playlist_video_ids]

for cand_id, name in valid_candidate_list:
    print("Candidate id:", cand_id)
    print("Title:", name)
    print('\n')

Candidate id: candidate_001
Title: Five Useful Redstone Farms for Survival Worlds


Candidate id: candidate_002
Title: Automatic Wheat Farm Tutorialâ€”Updated Design


Candidate id: candidate_003
Title: Three Cozy Minecraft Starter Bases


Candidate id: candidate_004
Title: Minecraft Hardcore Livestream: Building for Six Hours


Candidate id: candidate_005
Title: Fresh Lo-Fi Beats for Deep Focus


Candidate id: candidate_006
Title: Jazz Piano Improvisation for a Quiet Morning


Candidate id: candidate_007
Title: Build a Video Metadata Analyzer in Python


Candidate id: candidate_008
Title: Python Machine Learning in Ten Minutes


Candidate id: candidate_009
Title: Coding a Minecraft Inventory Tool with Python




In [20]:
# which candidates are relevant to more than one topic (tag)
# look at the list of tags, and format

import re

stop_words = {
    "a", "an", "the", "and", "or", "but", "for", "from",
    "in", "into", "of", "on", "to", "with", "by", "at",
    "as", "is", "are", "be", "this", "that", "these",
    "those", "your", "you", "my", "we", "it", "how", "what",
}
def extract_keywords(video, stop_words):
    tags_text = " ".join(video["tags"]) if video.get("tags") else ""
    
    text = f"{video['title']} {video['description']} {tags_text}"
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {word for word in words if word not in stop_words}


playlist_keywords_dict = {
    video["video_id"] : extract_keywords(video, stop_words)
    for video in playlist_videos
}

candidate_keywords_dict = {
    video["video_id"] : extract_keywords(video, stop_words)
    for video in candidate_videos
}

        

In [21]:
# find similarity score
def jaccard_similarity(set1, set2):
    if not set1 and not set2:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

    
candidate_scores = []
for cand_id, name in valid_candidate_list:
    cand_keywords = candidate_keywords_dict[cand_id]
    max_score = 0.0
    best_match_title = ""
    for playlist_vid in playlist_videos:
        p_id = playlist_vid["video_id"]
        p_keywords = playlist_keywords_dict[p_id]
        score = jaccard_similarity(cand_keywords, p_keywords)
        if score > max_score:
            max_score = score
            best_match_title = playlist_vid["title"]
    candidate_scores.append({"candidate_id": cand_id, "title": name, 
                             "max_similarity_score": max_score,
                             "matched_with": best_match_title})

candidate_scores.sort(key=lambda x: x["max_similarity_score"], reverse=True)
print("--- Video Recommendations ---")
for candidate in candidate_scores:
    print(f"Score: {candidate['max_similarity_score']:.3f} | {candidate['title']} | Matched with: {candidate["matched_with"]}")

--- Video Recommendations ---
Score: 0.467 | Automatic Wheat Farm Tutorialâ€”Updated Design | Matched with: Build an Automatic Wheat Farm in Minecraft
Score: 0.391 | Fresh Lo-Fi Beats for Deep Focus | Matched with: Lo-Fi Beats for Evening Study
Score: 0.333 | Five Useful Redstone Farms for Survival Worlds | Matched with: Build an Automatic Wheat Farm in Minecraft
Score: 0.227 | Coding a Minecraft Inventory Tool with Python | Matched with: Build a Small Python Project with JSON Data
Score: 0.217 | Build a Video Metadata Analyzer in Python | Matched with: Build a Small Python Project with JSON Data
Score: 0.130 | Three Cozy Minecraft Starter Bases | Matched with: Minecraft Redstone Doors Explained
Score: 0.130 | Jazz Piano Improvisation for a Quiet Morning | Matched with: Lo-Fi Beats for Evening Study
Score: 0.095 | Minecraft Hardcore Livestream: Building for Six Hours | Matched with: Build an Automatic Wheat Farm in Minecraft
Score: 0.074 | Python Machine Learning in Ten Minutes | Match